# Demo fine-tune Spark-TTS 0.5B cho tiếng Việt

Notebook gồm 3 phần:
1. **Chạy thử tiếng Việt trên model gốc** — nghe thử
2. **Fine-tune** bằng dataset `doof-ferb/infore1_25hours` (LoRA với Unsloth)
3. **Chạy lại** cùng các câu đó để so sánh trước / sau

> Runtime → Change runtime type → **GPU** (T4 miễn phí là đủ).

## 0. Cài đặt

In [1]:
%%capture
!pip install unsloth
!pip install "datasets>=3.4.1,<4.0.0" omegaconf einx soundfile
!git clone https://github.com/SparkAudio/Spark-TTS

## 0.1 Tải model và audio tokenizer (BiCodec)

In [2]:
import sys, re, torch, numpy as np
from unsloth import FastModel
from huggingface_hub import snapshot_download
from IPython.display import Audio, display

# Tải toàn bộ Spark-TTS (gồm phần LLM + BiCodec)
snapshot_download("unsloth/Spark-TTS-0.5B", local_dir="Spark-TTS-0.5B")

max_seq_length = 2048
model, tokenizer = FastModel.from_pretrained(
    model_name="Spark-TTS-0.5B/LLM",
    max_seq_length=max_seq_length,
    dtype=torch.float32,   # Spark hiện chỉ chạy ổn với float32
    full_finetuning=False,
    load_in_4bit=False,
)

# Audio tokenizer: chuyển audio <-> token
sys.path.append("Spark-TTS")
from sparktts.models.audio_tokenizer import BiCodecTokenizer
from sparktts.utils.audio import audio_volume_normalize
audio_tokenizer = BiCodecTokenizer("Spark-TTS-0.5B", "cuda")
SAMPLE_RATE = audio_tokenizer.config["sample_rate"]  # 16000
print("Sample rate:", SAMPLE_RATE)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1116 14:31:45.979000 495 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1116 14:31:46.021000 495 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 31 files:   0%|          | 0/31 [00:00<?, ?it/s]

==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Missing tensor: mel_transformer.spectrogram.window
Missing tensor: mel_transformer.mel_scale.fb


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: Spark-TTS-0.5B/wav2vec2-large-xlsr-53
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sample rate: 16000


## 0.2 Hàm sinh giọng nói

In [3]:
@torch.inference_mode()
def generate_speech(text, mode="tts", temperature=0.8, top_k=50, top_p=1.0,
                    max_new_audio_tokens=2048, gender="female"):
    """
    mode="controllable": prompt chuẩn của Spark-TTS gốc (chọn giới tính, cao độ, tốc độ)
                         -> dùng để test model GỐC cho công bằng.
    mode="tts":          prompt giống lúc fine-tune -> dùng cho model ĐÃ FINE-TUNE.
    """
    torch.compiler.reset()
    if mode == "controllable":
        gender_id = {"female": 0, "male": 1}[gender]
        prompt = "".join([
            "<|task_controllable_tts|>", "<|start_content|>", text, "<|end_content|>",
            "<|start_style_label|>", f"<|gender_{gender_id}|>",
            "<|pitch_label_2|>", "<|speed_label_2|>", "<|end_style_label|>",
        ])
    else:
        prompt = "".join([
            "<|task_tts|>", "<|start_content|>", text, "<|end_content|>",
            "<|start_global_token|>",
        ])

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs, max_new_tokens=max_new_audio_tokens, do_sample=True,
        temperature=temperature, top_k=top_k, top_p=top_p,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
    )
    pred = tokenizer.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=False)[0]

    semantic = re.findall(r"<\|bicodec_semantic_(\d+)\|>", pred)
    if not semantic:
        print("Cảnh báo: model không sinh ra semantic token nào.")
        return np.zeros(SAMPLE_RATE // 2, dtype=np.float32)
    semantic_ids = torch.tensor([int(t) for t in semantic]).long().unsqueeze(0)

    glob = re.findall(r"<\|bicodec_global_(\d+)\|>", pred)
    global_ids = (torch.tensor([int(t) for t in glob]).long().unsqueeze(0)
                  if glob else torch.zeros((1, 1), dtype=torch.long))

    audio_tokenizer.device = "cuda"
    audio_tokenizer.model.to("cuda")
    wav = audio_tokenizer.detokenize(global_ids.to("cuda"), semantic_ids.to("cuda"))
    return np.asarray(wav, dtype=np.float32)

## 1. Chạy thử tiếng Việt trên model GỐC
Model gốc chủ yếu được train tiếng Anh + tiếng Trung, nên dự kiến phát âm tiếng Việt sẽ sai nhiều.

In [4]:
FastModel.for_inference(model)

# Thử nhanh 1 câu với model gốc
text = "Chào mừng các bạn đến với kênh Mì AI, học AI theo cách mì ăn liền."
display(Audio(generate_speech(text, mode="controllable"), rate=SAMPLE_RATE))

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 2. Fine-tune bằng `doof-ferb/infore1_25hours`

### 2.1 Tải và lọc dữ liệu

In [5]:
from datasets import load_dataset, Audio as HFAudio

NUM_SAMPLES = 2000        # số mẫu dùng để train (tăng lên nếu có thời gian)
MAX_SECONDS = 15          # bỏ clip quá dài

ds = load_dataset("doof-ferb/infore1_25hours", split="train")
ds = ds.rename_column("transcription", "text")
ds = ds.shuffle(seed=42).select(range(min(NUM_SAMPLES * 2, len(ds))))
ds = ds.cast_column("audio", HFAudio(sampling_rate=SAMPLE_RATE))
ds = ds.filter(
    lambda x: x["text"] is not None and len(x["text"].strip()) > 0
              and len(x["audio"]["array"]) <= MAX_SECONDS * SAMPLE_RATE,
    desc="Lọc dữ liệu",
)
ds = ds.select(range(min(NUM_SAMPLES, len(ds))))
print(ds)
print("Ví dụ:", ds[0]["text"])

README.md:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

data/train-00000-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  486MB            

data/train-00000-of-00015.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  485MB            

data/train-00001-of-00015.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  522MB            

... (24 outputs left out)

data/train-00014-of-00015.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14935 [00:00<?, ? examples/s]

Lọc dữ liệu:   0%|          | 0/4000 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'text'],
    num_rows: 2000
})
Ví dụ: ghi lại những khoảnh khắc đáng yêu phút giây ngọt ngào của con là niềm vui của các bậc cha mẹ


### 2.2 Chuyển audio thành token BiCodec (định dạng train của Spark-TTS)

In [6]:
def extract_wav2vec2_features(wavs):
    wav_np = wavs.squeeze(0).cpu().numpy()
    processed = audio_tokenizer.processor(wav_np, sampling_rate=16000,
                                          return_tensors="pt", padding=True)
    input_values = processed.input_values.to(audio_tokenizer.feature_extractor.device)
    out = audio_tokenizer.feature_extractor(input_values)
    hs = out.hidden_states
    return (hs[11] + hs[14] + hs[16]) / 3

@torch.no_grad()
def format_example(example):
    audio_array = np.asarray(example["audio"]["array"], dtype=np.float32)
    if audio_tokenizer.config["volume_normalize"]:
        audio_array = audio_volume_normalize(audio_array)
    ref_wav = audio_tokenizer.get_ref_clip(audio_array)

    wav_t = torch.from_numpy(audio_array).unsqueeze(0).float().to(audio_tokenizer.device)
    ref_t = torch.from_numpy(ref_wav).unsqueeze(0).float().to(audio_tokenizer.device)
    feat = extract_wav2vec2_features(wav_t)

    semantic_ids, global_ids = audio_tokenizer.model.tokenize(
        {"wav": wav_t, "ref_wav": ref_t, "feat": feat.to(audio_tokenizer.device)}
    )
    global_tokens = "".join(f"<|bicodec_global_{i}|>" for i in global_ids.squeeze().cpu().numpy())
    semantic_tokens = "".join(f"<|bicodec_semantic_{i}|>" for i in semantic_ids.squeeze().cpu().numpy())

    text = "".join([
        "<|task_tts|>", "<|start_content|>", example["text"], "<|end_content|>",
        "<|start_global_token|>", global_tokens, "<|end_global_token|>",
        "<|start_semantic_token|>", semantic_tokens, "<|end_semantic_token|>",
        "<|im_end|>",
    ])
    return {"text": text}

train_ds = ds.map(format_example, remove_columns=ds.column_names, desc="Tokenize audio")

# Giải phóng GPU cho lúc train
audio_tokenizer.model.cpu()
audio_tokenizer.feature_extractor.cpu()
torch.cuda.empty_cache()
print(train_ds[0]["text"][:300], "...")

Parameter 'function'=<function format_example at 0x7b176b65d300> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Tokenize audio:   0%|          | 0/2000 [00:00<?, ? examples/s]

<|task_tts|><|start_content|>ghi lại những khoảnh khắc đáng yêu phút giây ngọt ngào của con là niềm vui của các bậc cha mẹ<|end_content|><|start_global_token|><|bicodec_global_835|><|bicodec_global_2289|><|bicodec_global_2272|><|bicodec_global_3799|><|bicodec_global_3766|><|bicodec_global_2375|><|bi ...


### 2.3 Gắn LoRA và train

In [7]:
model = FastModel.get_peft_model(
    model,
    r=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
FastModel.for_training(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(166000, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
                (base_

In [10]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=8, # Nhỏ thì về 2
        gradient_accumulation_steps=1, # Nhỏ thì lên 4
        warmup_steps=10,
        num_train_epochs=5,      # 2000 mẫu x 2 epoch ~ 1250 bước. Học ngôn ngữ mới cần nhiều bước
        learning_rate=2e-4,
        fp16=False, bf16=False,  # Spark train ở float32
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
stats = trainer.train()
print(f"Thời gian train: {stats.metrics['train_runtime']/60:.1f} phút")

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 5 | Total steps = 1,250
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 70,385,664 of 577,019,776 (12.20% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,5.613541
20,5.617908
30,5.599893
40,5.548363
50,5.610239
60,5.529683
70,5.574413
80,5.552656
90,5.562531
100,5.533992


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1250/tokenizer_config.json.


Thời gian train: 7.8 phút


## 3. Chạy lại sau fine-tune và so sánh
Dùng đúng các câu test ở phần 1.

In [9]:
FastModel.for_inference(model)

# Cùng câu đã thử ở phần 1
text = "Chào mừng các bạn đến với kênh Mì AI, học AI theo cách mì ăn liền."
display(Audio(generate_speech(text, mode="tts"), rate=SAMPLE_RATE))

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [11]:
FastModel.for_inference(model)

# Cùng câu đã thử ở phần 1
text = "Chào mừng các bạn đến với kênh Mì AI, học AI theo cách mì ăn liền."
display(Audio(generate_speech(text, mode="tts"), rate=SAMPLE_RATE))

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## (Tùy chọn) Lưu model

In [ ]:
# Lưu LoRA adapter
model.save_pretrained("spark_tts_vi_lora")
tokenizer.save_pretrained("spark_tts_vi_lora")
# model.push_to_hub("ten-ban/spark-tts-vi-lora")  # nếu muốn đẩy lên Hugging Face